In [10]:
import cv2
import numpy as np
import os
import random

In [15]:
def ajouter_brouillard(image_or_path, intensite=0.5):
    """
    Ajoute un effet de brouillard sur une image.
    image_or_path : chemin (str / PathLike) OU numpy.ndarray (BGR uint8)
    intensite : float entre 0 et 1
    Retour : image_brouillard (numpy.uint8 BGR)
    """
    # Charger si on a reçu un chemin
    if isinstance(image_or_path, (str, os.PathLike)):
        image = cv2.imread(str(image_or_path))
        if image is None:
            raise FileNotFoundError(f"Impossible de charger l'image : {image_or_path}")
    elif isinstance(image_or_path, np.ndarray):
        image = image_or_path.copy()
    else:
        raise TypeError("image_or_path doit être un chemin (str/PathLike) ou un numpy.ndarray")

    # Normaliser l'image (valeurs entre 0 et 1)
    image = image.astype(np.float32) / 255.0

    # Générer un nuage de brouillard (bruit flou)
    hauteur, largeur = image.shape[:2]
    bruit = np.random.normal(loc=0.5, scale=0.5, size=(hauteur, largeur)).astype(np.float32)
    brouillard = cv2.GaussianBlur(bruit, (0, 0), sigmaX=max(1.0, hauteur/10), sigmaY=max(1.0, largeur/10))

    # Normaliser et étendre sur 3 canaux
    brouillard = cv2.normalize(brouillard, None, 0, 1, cv2.NORM_MINMAX)
    # Ajouter une dimension pour avoir (hauteur, largeur, 1), puis répéter
    brouillard = brouillard[:, :, np.newaxis]
    brouillard = np.repeat(brouillard, 3, axis=2)

    # Mélange linéaire entre image originale et brouillard
    image_brouillard = cv2.addWeighted(image, 1 - float(intensite), brouillard, float(intensite), 0)

    # Revenir à l'échelle 0—255
    image_brouillard = (np.clip(image_brouillard, 0.0, 1.0) * 255).astype(np.uint8)

    return image_brouillard

In [48]:
def traiter_images_random(train_dir, output_dir, n_images=250, intensite_max=0.5,intensite_min=0.3):
    """Sélectionne n_images aléatoires dans train_dir, applique le brouillard, et sauvegarde dans output_dir."""

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Lister toutes les images
    toutes_images = [f for f in os.listdir(train_dir)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]

    if len(toutes_images) == 0:
        print("⚠️ Aucun fichier image trouvé dans le dossier.")
        return

    # Sélection aléatoire
    images_selectionnees = random.sample(toutes_images, min(n_images, len(toutes_images)))
    intensite = random.uniform(intensite_min, intensite_max)
    for nom_fichier in images_selectionnees:
        chemin_entree = os.path.join(train_dir, nom_fichier)

        image = cv2.imread(chemin_entree)
        if image is None:
            print(f"⚠️ Impossible de lire {nom_fichier}")
            continue

        image_modifiee = ajouter_brouillard(image, intensite=intensite)
        nom_fichier = f"fog_{nom_fichier}"
        chemin_sortie = os.path.join(output_dir, nom_fichier)
        cv2.imwrite(chemin_sortie, image_modifiee)
        print(f"✅ Image traitée : {nom_fichier}")

    print(f"\n✅ Traitement terminé ! {len(images_selectionnees)} images modifiées sont dans : {output_dir}")


In [50]:
if __name__ == "__main__":
    dossier_train = "./dataset/images/train"
    dossier_prepare = "./train_prepare"
    traiter_images_random(dossier_train, dossier_prepare, n_images=10, intensite_max=0.7,intensite_min=0.2)

✅ Image traitée : fog_gss1121_jpg.rf.caee2cc7758458361a0f4ebcad31dd5a.jpg
✅ Image traitée : fog_gss634_jpg.rf.04e07dc34704292a72eb0e926116aec6.jpg
✅ Image traitée : fog_gss222_jpg.rf.9afdb243e708364184d48b6f5c7d94bd.jpg
✅ Image traitée : fog_gss945_jpg.rf.ce1f502561a27ff7576ffdff56185789.jpg
✅ Image traitée : fog_gss1127_jpg.rf.110ad7ce5a79357055c6a29c26a65af1.jpg
✅ Image traitée : fog_gss2018_jpg.rf.f5c4a747020f596243a2ba954c5dd52e.jpg
✅ Image traitée : fog_gss1271_jpg.rf.9ff5197ea7e22a0cf3c177b39cebc2b9.jpg
✅ Image traitée : fog_gss579_jpg.rf.59fe8e3f990baef6f44e059fbefcde40.jpg
✅ Image traitée : fog_gss732_jpg.rf.1943ffdf6019d22e7f40b59339ab2140.jpg
✅ Image traitée : fog_gss2104_jpg.rf.376f0ce047585574644db0d9d6d4def4.jpg

✅ Traitement terminé ! 10 images modifiées sont dans : ./train_prepare


In [51]:
import shutil
import os

# Chemin du dossier à supprimer
dossier_a_supprimer = "./train_prepare"

# Vérifier si le dossier existe
if os.path.exists(dossier_a_supprimer):
    # Supprimer le dossier et tout son contenu
    shutil.rmtree(dossier_a_supprimer)
    print(f"✅ Dossier '{dossier_a_supprimer}' supprimé avec succès !")
else:
    print(f"⚠️ Le dossier '{dossier_a_supprimer}' n'existe pas.")

✅ Dossier './train_prepare' supprimé avec succès !
